# Finding and fixing a noisy-neighbor problem in a multi-tenant Kubernetes cluster

Requires Elasticsearch and Kibana 9.4+, and the lab in `k8s/` (see `k8s/README.md`). Copy `.env.example` to `.env` and set an API key created in Kibana **Dev Tools**:

```
POST /_security/api_key
{ "name": "noisy-neighbor-notebook", "role_descriptors": {} }
```

Order: `./lab.sh up`, `./lab.sh baseline`, `./lab.sh probe` (keep it running), wait 5 minutes, `./lab.sh contention`, wait 8 minutes, run sections 1 to 5. Then `./lab.sh fix`, wait 8 minutes, run sections 6 and 7.

In [ ]:
%pip install -q -r requirements.txt

## 1. Connect

In [ ]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import requests
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

load_dotenv()

ES_URL = os.environ["ELASTICSEARCH_URL"]
API_KEY = os.environ["ELASTICSEARCH_API_KEY"]
KIBANA_URL = os.environ["KIBANA_URL"].rstrip("/")
SPACE = os.getenv("KIBANA_SPACE", "default")
KBN_BASE = KIBANA_URL if SPACE == "default" else f"{KIBANA_URL}/s/{SPACE}"

CLUSTER = os.getenv("LAB_CLUSTER_NAME", "noisy-neighbor")
NODE = os.getenv("LAB_NODE_NAME", "noisy-neighbor-worker")
TEST_ID = os.getenv("LAB_TEST_ID", "noisy-neighbor-final")
WINDOW_MINUTES = int(os.getenv("LAB_WINDOW_MINUTES", "2") or "2")
RANGE_FROM = os.getenv("LAB_RANGE_FROM", "").strip()
RANGE_TO = os.getenv("LAB_RANGE_TO", "").strip()

# Data streams written by the EDOT Collector and the k6 probe.
KUBELET_STREAM = "metrics-kubeletstatsreceiver.otel-*"
CLUSTER_STREAM = "metrics-k8sclusterreceiver.otel-*"
PROBE_STREAM = "metrics-generic.otel-*"

# Created by this notebook and deleted in Clean up.
SLO_ID = "noisy-neighbor-storefront"

es = Elasticsearch(ES_URL, api_key=API_KEY, request_timeout=120)
KBN_HEADERS = {
    "Authorization": f"ApiKey {API_KEY}",
    "kbn-xsrf": "true",
    "Content-Type": "application/json",
}


def kbn(method, path, **kwargs):
    """Call a Kibana API in the configured space and raise on HTTP errors."""
    response = requests.request(
        method, f"{KBN_BASE}{path}", headers=KBN_HEADERS, timeout=120, **kwargs
    )
    response.raise_for_status()
    return response.json() if response.text else {}


def esql(query, title=None):
    """Run an ES|QL query and print the result as a text table."""
    response = es.esql.query(query=query, format="txt")
    if title:
        print(f"\n{title}")
    print(response.body)
    return response.body


def window(minutes=None):
    """WHERE fragment for the query window: fixed bounds from .env, or the last N minutes."""
    if RANGE_FROM and RANGE_TO:
        return f'@timestamp >= "{RANGE_FROM}" AND @timestamp <= "{RANGE_TO}"'
    return f"@timestamp >= NOW() - {minutes or WINDOW_MINUTES} minutes"


print(
    f"elasticsearch {es.info()['version']['number']} | space={SPACE} | window: {window()}"
)
streams = es.indices.get_data_stream(name="metrics-*.otel-*")["data_streams"]
names = sorted(s["name"] for s in streams)
print("OTel metric data streams:", ", ".join(names) or "none")
missing = []
for pattern in (KUBELET_STREAM, CLUSTER_STREAM, PROBE_STREAM):
    prefix = pattern.rstrip("*")
    found = [n for n in names if n.startswith(prefix)]
    print(f"  {pattern}: {'ok ' + ', '.join(found) if found else 'MISSING'}")
    if not found:
        missing.append(pattern)
if missing:
    raise RuntimeError(
        f"missing {missing}: run ./lab.sh up and ./lab.sh probe, wait two minutes, and rerun this cell"
    )

## 2. Check the tenant labels

A null tenant is an unlabeled workload.

In [ ]:
esql(
    f"""
FROM {KUBELET_STREAM}
| WHERE {window()}
    AND k8s.cluster.name == "{CLUSTER}"
    AND k8s.node.name == "{NODE}"
    AND k8s.pod.name IS NOT NULL
| STATS pods = COUNT_DISTINCT(k8s.pod.name)
    BY k8s.namespace.name, resource.attributes.tenant, resource.attributes.app
| SORT k8s.namespace.name
| LIMIT 50
""",
    "Pods on the worker by namespace, tenant and app",
)

## 3. Confirm the latency symptom

Storefront p95 per minute, then restarts and phase (2 = Running).

In [ ]:
esql(
    f"""
FROM {PROBE_STREAM}
| WHERE {window(30)}
    AND service.name == "noisy-neighbor-probe"
    AND attributes.tenant == "storefront"
| STATS p95_ms = PERCENTILE(k6_http_req_duration, 95)
    BY minute = BUCKET(@timestamp, 1 minute), tenant = attributes.tenant
| EVAL p95_ms = ROUND(p95_ms, 1)
| SORT minute
| LIMIT 100
""",
    "Storefront p95 latency per minute",
)

esql(
    f"""
FROM {CLUSTER_STREAM}
| WHERE {window(30)}
    AND k8s.cluster.name == "{CLUSTER}"
    AND k8s.namespace.name == "team-storefront"
    AND k8s.pod.name LIKE "storefront-*"
| STATS restarts = MAX(k8s.container.restarts), phase = MAX(k8s.pod.phase)
    BY k8s.pod.name
| LIMIT 20
""",
    "Storefront pod restarts and phase (2 = Running)",
)

## 4. Rank every tenant on the node

In [ ]:
RANK_QUERY = f"""
TS {KUBELET_STREAM}
| WHERE {window()}
    AND k8s.cluster.name == "{CLUSTER}"
    AND k8s.node.name == "{NODE}"
| STATS cpu_cores = SUM(AVG_OVER_TIME(k8s.pod.cpu.usage)),
        memory_bytes = SUM(AVG_OVER_TIME(k8s.pod.memory.working_set)),
        cpu_limit_used = MAX(AVG_OVER_TIME(k8s.pod.cpu_limit_utilization))
    BY k8s.node.name, k8s.namespace.name, resource.attributes.tenant
| EVAL cpu_cores = ROUND(cpu_cores, 3),
       memory_mib = ROUND(memory_bytes / 1048576.0, 1),
       cpu_limit_used_pct = ROUND(cpu_limit_used * 100, 0)
| DROP memory_bytes, cpu_limit_used
| SORT cpu_cores DESC
| RENAME k8s.node.name AS node, k8s.namespace.name AS namespace,
    resource.attributes.tenant AS tenant
| LIMIT 5
"""

NODE_QUERY = f"""
TS {KUBELET_STREAM}
| WHERE {window()}
    AND k8s.cluster.name == "{CLUSTER}"
    AND k8s.node.name == "{NODE}"
| STATS node_cpu_cores = MAX(AVG_OVER_TIME(k8s.node.cpu.usage)) BY k8s.node.name
| EVAL node_cpu_cores = ROUND(node_cpu_cores, 2)
"""

esql(RANK_QUERY, "Tenants on the worker, ranked by CPU")
esql(NODE_QUERY, "Worker CPU usage")

## 5. Trace the tenant to its Deployment

In [ ]:
esql(
    f"""
TS {KUBELET_STREAM}
| WHERE {window()}
    AND k8s.cluster.name == "{CLUSTER}"
    AND k8s.namespace.name == "team-batch"
| STATS cpu_cores = SUM(AVG_OVER_TIME(k8s.pod.cpu.usage)),
        memory_bytes = SUM(AVG_OVER_TIME(k8s.pod.memory.working_set))
    BY resource.attributes.app, k8s.deployment.name, k8s.node.name
| EVAL cpu_cores = ROUND(cpu_cores, 3),
       memory_mib = ROUND(memory_bytes / 1048576.0, 1)
| DROP memory_bytes
| SORT cpu_cores DESC
| RENAME resource.attributes.app AS app,
    k8s.deployment.name AS deployment, k8s.node.name AS node
| LIMIT 10
""",
    "team-batch by app label and Deployment",
)

esql(
    f"""
TS {KUBELET_STREAM}
| WHERE {window()}
    AND k8s.cluster.name == "{CLUSTER}"
    AND k8s.node.name == "{NODE}"
    AND k8s.namespace.name IN ("team-storefront", "team-batch")
| STATS cpu_request_used = MAX(AVG_OVER_TIME(k8s.container.cpu_request_utilization))
    BY k8s.namespace.name, k8s.deployment.name, k8s.container.name
| EVAL cpu_request_used_pct = ROUND(cpu_request_used * 100, 0)
| DROP cpu_request_used
| SORT cpu_request_used_pct DESC
| LIMIT 10
""",
    "Container CPU use as a percentage of its request",
)

## 6. Verify the fix

Run after `./lab.sh fix`.

In [ ]:
esql(RANK_QUERY, "Tenants on the worker after the fix")
esql(NODE_QUERY, "Worker CPU usage after the fix")

esql(
    f"""
FROM {PROBE_STREAM}
| WHERE {window(60)}
    AND service.name == "noisy-neighbor-probe"
    AND attributes.tenant == "storefront"
| STATS median_ms = PERCENTILE(k6_http_req_duration, 50),
        p95_ms = PERCENTILE(k6_http_req_duration, 95)
    BY minute = BUCKET(@timestamp, 1 minute)
| EVAL median_ms = ROUND(median_ms, 1), p95_ms = ROUND(p95_ms, 1)
| SORT minute
| LIMIT 100
""",
    "Storefront median and p95 per minute across baseline, contention and fix",
)

## 7. Create the latency SLO

99% of probe requests with HTTP 200 under 500 ms, over a rolling 30 days. Waits 150 seconds for the first result.

In [ ]:
slo_body = {
    "id": SLO_ID,
    "name": "Storefront fairness: 99% successful probes under 500 ms",
    "description": "Track storefront HTTP probes while batch work shares its node.",
    "indicator": {
        "type": "sli.histogram.custom",
        "params": {
            "index": PROBE_STREAM,
            "timestampField": "@timestamp",
            "filter": f'service.name: "noisy-neighbor-probe" and attributes.test-id: "{TEST_ID}"',
            "good": {
                "field": "k6_http_req_duration",
                "aggregation": "range",
                "from": 0,
                "to": 500,
                "filter": 'attributes.status: "200"',
            },
            "total": {"field": "k6_http_req_duration", "aggregation": "value_count"},
        },
    },
    "budgetingMethod": "occurrences",
    "timeWindow": {"duration": "30d", "type": "rolling"},
    "objective": {"target": 0.99},
    "groupBy": ["attributes.tenant"],
    "settings": {"syncDelay": "1m", "frequency": "1m"},
    "tags": ["noisy-neighbor-lab", "kubernetes", "resource-fairness"],
}

existing = requests.get(
    f"{KBN_BASE}/api/observability/slos/{SLO_ID}", headers=KBN_HEADERS, timeout=60
)
if existing.status_code == 200:
    print(f"SLO {SLO_ID} already exists, reusing it")
else:
    created = kbn("POST", "/api/observability/slos", data=json.dumps(slo_body))
    print(f"created SLO {created['id']}")

definition = kbn("GET", f"/api/observability/slos/{SLO_ID}")
print(
    json.dumps(
        {
            k: definition[k]
            for k in (
                "name",
                "budgetingMethod",
                "timeWindow",
                "objective",
                "groupBy",
                "settings",
            )
        },
        indent=2,
    )
)
print(
    "indicator:",
    definition["indicator"]["type"],
    "| index:",
    definition["indicator"]["params"]["index"],
)

print("\nwaiting 150 s for the sync delay and the first transform run...")
time.sleep(150)
summary = kbn(
    "GET", f"/api/observability/slos/{SLO_ID}", params={"instanceId": "storefront"}
).get("summary", {})
print(
    "status:",
    summary.get("status"),
    "| SLI:",
    summary.get("sliValue"),
    "| error budget remaining:",
    summary.get("errorBudget", {}).get("remaining"),
)
print("Kibana:", f"{KBN_BASE}/app/slos/{SLO_ID}?instanceId=storefront")

## 8. Clean up

Deletes the SLO. `./lab.sh down` deletes the cluster.

In [ ]:
response = requests.delete(
    f"{KBN_BASE}/api/observability/slos/{SLO_ID}", headers=KBN_HEADERS, timeout=120
)
print(f"delete SLO {SLO_ID}: HTTP {response.status_code}")
print("done")